# Tutorial: CryptoHFTData Apex Backtest Workflow

Audience:
- Apex users who want to prepare CryptoHFTData historical files and run a small backtest against the prepared data.

Prerequisites:
- Apex has been built locally, or CMake can build the example from this notebook.
- Python dependencies from `python/requirements-cryptohftdata.txt` are installed.
- `CRYPTOHFTDATA_API_KEY` is set, or you are ready to enter it when prompted. The key is never printed.

Learning goals:
- Prepare CryptoHFTData `trades` and `orderbook` files into Apex `tickbin1` data.
- Inspect the preparation manifest and a few generated records.
- Run the `apex-example-cryptohftdata-backtest` market-making example on the prepared range.


## Outline

1. Configure the venue, symbol, and Apex-style half-open time range.
2. Generate Apex reference data if it is missing.
3. Run the CryptoHFTData converter.
4. Inspect the manifest and prepared `tickbin1` files.
5. Build or locate the C++ backtest example.
6. Run a simple market-making backtest.
7. Try a small extension.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "CMakeLists.txt").exists():
    REPO_ROOT = REPO_ROOT.parent

APEX_HOME = Path(os.environ.get("APEX_HOME", Path.home() / "apex")).expanduser()
TICKDATA_DIR = APEX_HOME / "data" / "tickdata"

VENUE = os.environ.get("APEX_CHD_VENUE", "binance_usdfut")
SYMBOL = os.environ.get("APEX_CHD_SYMBOL", "BTCUSDT")
START = os.environ.get("APEX_CHD_FROM", "2026-05-03T00:00:00")
UPTO = os.environ.get("APEX_CHD_UPTO", "2026-05-04T00:00:00")
STREAMS = "aggtrades,l1"

api_key = os.environ.get("CRYPTOHFTDATA_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CryptoHFTData API key: ").strip()
    if api_key:
        os.environ["CRYPTOHFTDATA_API_KEY"] = api_key
if not api_key:
    raise RuntimeError("CRYPTOHFTDATA_API_KEY is required to download CryptoHFTData files")

print("repo:", REPO_ROOT)
print("APEX_HOME:", APEX_HOME)
print("range:", f"[{START}, {UPTO})")
print("dataset:", VENUE, SYMBOL, STREAMS)
print("has API key:", bool(api_key))


## 1. Preflight

This cell checks that the converter and optional Python dependencies are visible. The API key is never printed.


In [ ]:
converter = REPO_ROOT / "python" / "prepare_cryptohftdata.py"
requirements = REPO_ROOT / "python" / "requirements-cryptohftdata.txt"

print("converter exists:", converter.exists(), converter)
print("requirements file:", requirements)
print("has API key:", bool(os.environ.get("CRYPTOHFTDATA_API_KEY")))

def missing_packages() -> list[str]:
    missing = []
    for package in ("pyarrow", "zstandard"):
        try:
            __import__(package)
        except ImportError:
            missing.append(package)
    return missing

missing = missing_packages()
if missing:
    print("installing missing parquet dependencies:", ", ".join(missing))
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-r",
            str(requirements),
        ],
        check=True,
    )
    missing = missing_packages()

if missing:
    raise RuntimeError("Missing parquet dependencies after install: " + ", ".join(missing))
print("parquet dependencies are installed")


## 2. Ensure Apex Refdata

The notebook generates Apex reference data automatically if `instruments.csv` is not present under `APEX_HOME`.


In [ ]:
refdata_csv = APEX_HOME / "data" / "refdata" / "instruments" / "instruments.csv"

print("refdata:", refdata_csv)
print("refdata exists:", refdata_csv.exists())

if not refdata_csv.exists():
    env = os.environ.copy()
    env["APEX_HOME"] = str(APEX_HOME)
    completed = subprocess.run(
        ["bash", str(REPO_ROOT / "python" / "generate-refdata.sh")],
        cwd=REPO_ROOT / "python",
        env=env,
        text=True,
        capture_output=True,
        check=True,
    )
    if completed.stdout:
        print(completed.stdout[-4000:])
    if completed.stderr:
        print(completed.stderr[-4000:])
    if not refdata_csv.exists():
        raise RuntimeError(f"refdata generation did not create {refdata_csv}")
else:
    print("using existing refdata")


## 3. Prepare CryptoHFTData for Apex

The converter uses Apex-style half-open time semantics: `[from, upto)`. Raw hourly CryptoHFTData files are deleted after successful conversion unless `--keep-raw` is supplied.


In [ ]:
prepare_cmd = [
    sys.executable,
    str(converter),
    "--venues", VENUE,
    "--symbols", SYMBOL,
    "--streams", STREAMS,
    "--from", START,
    "--upto", UPTO,
    "--jobs", "4",
    "--overwrite",
]

print(" ".join(prepare_cmd))

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "python") + os.pathsep + env.get("PYTHONPATH", "")
try:
    completed = subprocess.run(
        prepare_cmd,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
        check=True,
    )
except subprocess.CalledProcessError as exc:
    if exc.stdout:
        print(exc.stdout[-4000:])
    if exc.stderr:
        print(exc.stderr[-4000:])
    raise

if completed.stderr:
    print(completed.stderr[-4000:])
prepare_result = json.loads(completed.stdout)
print("manifest:", prepare_result["manifest"])
print("downloads:", prepare_result["downloads"])
for item in prepare_result["outputs"]:
    print(
        f"{item['venue']} {item['symbol']} {item['stream']} "
        f"emitted={item['emitted']} warnings={item.get('warning_count', len(item['warnings']))}"
    )


## 4. Inspect the Manifest

After preparation, this cell loads the manifest that records downloaded files, conversion counts, warning counts, output paths, and verification results.


In [ ]:
manifest_path = Path(prepare_result["manifest"])
if not manifest_path.exists():
    raise RuntimeError(f"manifest not found: {manifest_path}")

manifest = json.loads(manifest_path.read_text())
print("manifest:", manifest_path)
print("raw deleted after success:", manifest["raw_deleted_after_success"])
print("outputs:")
for item in manifest["outputs"]:
    print(
        f"  {item['venue']} {item['symbol']} {item['stream']} {item['day']} "
        f"emitted={item['emitted']} warnings={item.get('warning_count', len(item['warnings']))}"
    )


## 5. Inspect Prepared `tickbin1` Records

This uses the Python tickbin helper to read a few records from the generated Apex files without starting the C++ engine.


In [ ]:
sys.path.insert(0, str(REPO_ROOT / "python"))
from apex.cryptohftdata.tickbin import iter_records

if manifest_path is None:
    print("No manifest available.")
else:
    for item in manifest["outputs"]:
        path = Path(item["path"])
        stream = item["stream"]
        print("\n", stream, path)
        if not path.exists():
            print("  missing output file")
            continue
        for idx, record in enumerate(iter_records(path, stream)):
            print(" ", record)
            if idx >= 2:
                break


## 6. Build or Locate the Backtest Example

The C++ example is `apex-example-cryptohftdata-backtest`. It reads `APEX_CHD_VENUE`, `APEX_CHD_SYMBOL`, `APEX_CHD_FROM`, and `APEX_CHD_UPTO`, so the notebook and executable use the same range. If the executable is missing, this cell builds it.


In [ ]:
def run_checked(command: list[str], *, cwd: Path) -> subprocess.CompletedProcess[str]:
    print(" ".join(str(part) for part in command))
    try:
        result = subprocess.run(
            command,
            cwd=cwd,
            text=True,
            capture_output=True,
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        if exc.stdout:
            print(exc.stdout[-4000:])
        if exc.stderr:
            print(exc.stderr[-4000:])
        raise
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    return result

BUILD_DIR = Path(os.environ.get("APEX_CHD_BUILD_DIR", REPO_ROOT / "BUILD" / "cryptohft-debug"))
BUILD_CANDIDATES = [
    BUILD_DIR,
    REPO_ROOT / "BUILD" / "debug",
    REPO_ROOT / "BUILD-DEBUG",
]

exe_rel = Path("src/examples/standalone/apex-example-cryptohftdata-backtest")
example_exe = next((root / exe_rel for root in BUILD_CANDIDATES if (root / exe_rel).exists()), None)

if example_exe:
    print("example executable:", example_exe)
else:
    print("example executable not found; building it in", BUILD_DIR)
    if not (BUILD_DIR / "CMakeCache.txt").exists():
        run_checked(
            [
                "cmake",
                "-S", str(REPO_ROOT),
                "-B", str(BUILD_DIR),
                "-DCMAKE_BUILD_TYPE=Debug",
            ],
            cwd=REPO_ROOT,
        )
    run_checked(
        [
            "cmake",
            "--build", str(BUILD_DIR),
            "--target", "apex-example-cryptohftdata-backtest",
            "-j", "4",
        ],
        cwd=REPO_ROOT,
    )
    example_exe = BUILD_DIR / exe_rel
    if not example_exe.exists():
        raise RuntimeError(f"example executable was not built: {example_exe}")
    print("example executable:", example_exe)


## 7. Run the Market-Making Backtest

This runs the standalone Apex backtest over the same half-open range. It expects both `aggtrades` and `l1` files to exist under `$APEX_HOME/data/tickdata/tickbin1`.


In [ ]:
if not example_exe:
    raise RuntimeError("No executable found. The build cell should have created it.")

env = os.environ.copy()
env.update({
    "APEX_CHD_VENUE": VENUE,
    "APEX_CHD_SYMBOL": SYMBOL,
    "APEX_CHD_FROM": START,
    "APEX_CHD_UPTO": UPTO,
})
completed = subprocess.run(
    [str(example_exe)],
    cwd=REPO_ROOT,
    env=env,
    text=True,
    capture_output=True,
    check=False,
)
print("return code:", completed.returncode)
print(completed.stdout[-4000:])
if completed.stderr:
    print(completed.stderr[-4000:])
if completed.returncode != 0:
    raise RuntimeError(f"backtest failed with return code {completed.returncode}")


## Exercise

Change `VENUE`, `SYMBOL`, `START`, and `UPTO` to prepare a short Bybit range. Keep the range small at first, such as one hour, then compare the manifest counts with the Binance run.

Answer scaffold:
- Set `APEX_CHD_VENUE=bybit`.
- Pick a Bybit symbol that appears in CryptoHFTData symbol discovery, such as `BTCUSDT`.
- Re-run the preparation and manifest cells.
- Re-run the backtest with the same environment variables.


## Common Pitfalls

- If the backtest says no tick files were found, check that the manifest output path matches `$APEX_HOME/data/tickdata/tickbin1/{venue}/{stream}/YYYY/MM/DD/{symbol}.bin`.
- If symbol validation fails, query CryptoHFTData symbols for the venue/data type and use the native symbol exactly.
- If Apex cannot find the instrument, regenerate reference data with `python/generate-refdata.sh` and confirm the venue is one Apex already supports.
- If files already exist, rerun the converter with `--overwrite` only when replacing that prepared data is intentional.
